This Jupyter notebook is a variation of the VehicleType_Visualization_Bokeh notebook.  It is the same data but provides a method to change the colors of the sections.  It was a neat exercise.

In [ ]:
import os
import sqlite3
import pandas as pd
from bokeh.io import output_file, show
from bokeh.plotting import figure
from bokeh.transform import cumsum
from bokeh.palettes import Category20c, Category20
from bokeh.models import ColumnDataSource
from math import pi
import panel as pn
from panel.io import save
import panel.io as pio

In [ ]:
# Connect to SQLite database
cwd = os.getcwd()
conn = sqlite3.connect(f'{cwd}/data/crash_data.db')# Connect to SQLite database

# Execute SQL query and load results into a DataFrame
query = """
SELECT DISTINCT(UnitType), COUNT(*) as count
FROM ksp_vehicles
GROUP BY UnitType
HAVING COUNT(*) > 1
"""
df = pd.read_sql_query(query, conn)

# Close the database connection
conn.close()

In [ ]:
# Prepare data for the pie chart
df['angle'] = df['count'] / df['count'].sum() * 2 * pi
df['color'] = Category20c[len(df)]  # Use Category20 which has 20 colors

# Create ColumnDataSource
source = ColumnDataSource(df)

In [ ]:
# Create a Bokeh pie chart
p = figure(height=550, width=820, title="Vehicle Types Involved in Work Zone Collisions (2020-2024)",
           toolbar_location=None, tools="hover",
           tooltips="@UnitType: @count", x_range=(-0.5, 1.0))

r = p.wedge(x=0, y=1, radius=0.4,
            start_angle=cumsum('angle', include_zero=True), end_angle=cumsum('angle'),
            line_color="white", fill_color='color', legend_field='UnitType', source=source)

p.axis.axis_label = None
p.axis.visible = False
p.grid.grid_line_color = None

# Adjust legend properties
p.legend.title = 'Vehicle Type'
p.legend.location = 'top_right'
p.legend.orientation = 'vertical'
p.legend.label_text_font_size = '10pt'

In [ ]:
# Integrate Panel to allow for dynamic updates
#pn.extension()
#bokeh_pane = pn.pane.Bokeh(p)

# Integrate Panel to allow for dynamic updates
pn.extension()
bokeh_pane = pn.pane.Bokeh(p, sizing_mode='stretch_both')

In [ ]:
# Function to dynamically update colors
def update_colors(event):
    new_colors = Category20[len(df)]
    source.data['color'] = new_colors
    bokeh_pane.param.trigger('object')  # Refresh the plot

In [ ]:
# Create a button to trigger the color update
button = pn.widgets.Button(name='Update Colors', button_type='primary')
button.on_click(update_colors)

In [ ]:
# Layout
layout = pn.Column(bokeh_pane, button, sizing_mode='stretch_both')

# Save the layout as an HTML file
#pio.save(layout, 'vehicle_types_with_button.html')

# To display the plot with the button in a notebook, use the following:
layout.servable()

# Alternatively, if you are running a Bokeh server, use:
# pn.serve(layout)